### Responsibilities
- Load the cleaned dataset.
- Create new analysis features such as year, month, quarter, weekday, weekend flag, and profit margin.
- Save the processed dataset for SQL analysis.

### Deliverables
- `processed_sales_data.csv`
- `02_feature_engineering.ipynb`
- Feature engineering notebook
- Explanation of new columns

## Derived Columns

| Column Name | Description | Type |
|---|---|---|
| year | Year extracted from order_date | Numeric |
| month | Month name extracted from order_date | Text |
| month_num | Month number extracted from order_date | Numeric |
| quarter | Quarter extracted from order_date | Text |
| day_of_week | Day name extracted from order_date | Text |
| weekend_flag | Indicates whether the order was placed on a weekend | Boolean |
| profit_margin | Profit divided by sales | Numeric |
| sales_band | Sales grouped into low, medium, or high ranges | Text |

## Notes
- `profit_margin` should be handled safely to avoid division by zero.
- `order_date` and `ship_date` should be converted to proper date format before extracting features.
- Column names should be standardized to lowercase with underscores.
- Raw data should remain unchanged in `data/raw/`.
- Cleaned and processed files should be saved in `data/processed/`.


## 5. Main Questions
- Which categories generate the highest sales and profit? - categories, sales, and profit
- Which regions perform best? city, state, region
- Which products are most profitable?
- How do sales change over time?
- What is the effect of discounts on profit?
- Which customer segments contribute most to revenue?

In [110]:
import pandas as pd
import numpy as np

In [111]:
df = pd.read_csv('../data/cleaned/cleaned_data.csv', parse_dates=['order_date', 'ship_date'])

In [112]:
# Extracting year, month, and day from order_date
df['order_year'] = df['order_date'].dt.year 
df['order_month'] = df['order_date'].dt.month_name()
df['order_month_num'] = df['order_date'].dt.month

# Extracting day of the week from order_date
df['order_day_of_week'] = df['order_date'].dt.day_name()

In [113]:
# weekend_flag | Indicates whether the order was placed on a weekend (True for weekend, False for others)
df['weekend_flag'] = df['order_day_of_week'].isin(['Saturday', 'Sunday'])

## Extract quarter

In [114]:
# quarter | Quarter extracted from order_date | Text |
df['quarter'] = df['order_date'].dt.quarter

# Calculate profit margin
$$\text{Profit Margin} = \left( \frac{\text{Net Profit}}{\text{Net Sales}} \right) \times 100$$

In [115]:
# A much faster approach would be to use numpy because it uses vectorization
df['profit_margin'] = np.where(df['sales'] > 0, ((df['profit'])/df['sales'])*100, 0)

## Creating the sales bands with percentiles
- Subdividing the long tail
percentile_bins = [0, .25, .50, .75, .90, .95, 100]
percentile_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High', 'Premium']
use pd.qcut()

In [116]:
percentile_bins = [0, 0.25, 0.50, 0.75, 0.90, 0.95, 1.0]
percentile_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High', 'Premium']

df['sales_band'] = pd.qcut(df['sales'], q=percentile_bins, labels=percentile_labels)

In [117]:
# range_comparison = df.groupby('sales_band_percentile')['sales'].agg(['min', 'max', 'count'])
# print(range_comparison)

In [118]:
df.head(5)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,discount,profit,order_year,order_month,order_month_num,order_day_of_week,weekend_flag,quarter,profit_margin,sales_band
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,November,11,Tuesday,False,4,16.00,High
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,November,11,Tuesday,False,4,30.00,Very High
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,June,6,Sunday,True,2,47.00,Very Low
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,October,10,Sunday,True,4,-40.00,Premium
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,October,10,Sunday,True,4,11.25,Low


In [ ]:
isna = ['']

In [ ]:
# Save processed file to csv
df.to_csv('../data/cleaned/processed_sales_data.csv')